# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. You'll learn how to examine, process, and visualize data defined by a Croissant schema.

### Dataset Source
This dataset comes from the following Croissant schema URL:

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Let's load the FAIR² metadata and available records using `mlcroissant`. This reveals high-level information and allows access to each data 'record set'.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the metadata with mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Name: {metadata.name}\n\nDescription: {metadata.description}\n")

## 2. Data Overview
Review dataset record sets (tables), fields, and field `@id`s.

<br>
**Each entity (record set, field, column) is referenced by its `@id`.**

Let's print the record sets with their `@id` and the `@id` for their fields.

In [ ]:
# Inspect record sets in the dataset

record_sets = list(dataset.record_sets)
print(f"Total record sets: {len(record_sets)}\n")

for record_set in record_sets:
    print(f"RecordSet @id: {record_set.id}")
    print(f"  Title: {record_set.name}")
    print(f"  Description: {record_set.description}")
    print("  Field @ids:")
    for field in record_set.fields:
        print(f"    - {field.id} (name: {field.name}, type: {field.data_type})")
    print()

## 3. Data Extraction
Let's extract all data from a chosen record set into a DataFrame.

**Choose the main record set for analysis (`@id`). Replace the value with the appropriate `@id` from the record sets above if desired.**

In [ ]:
# Example: extract from the main protein quantification record set. Choose your RecordSet @id.
# Let's print all available record set @ids first for guidance.
record_set_ids = [rs.id for rs in record_sets]
print('Record set @ids:', record_set_ids)

# Choose a record set (replace below if you wish):
main_record_set_id = record_set_ids[0]  # Use the first as default example

# List all records for each record set, store as DataFrames
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(records)} records from RecordSet {rs_id}")

df = dataframes[main_record_set_id]
print("\nColumns for record set", main_record_set_id, ":\n", df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and group records to discover data patterns.

Below, supply the relevant field `@id`s (from the fields printed above) for numeric and grouping analysis.

In [ ]:
# Choose a numeric field and group field by their @id (replace values based on record set fields printed earlier)

# Example: find a numeric field for peptide count, e.g. '@id': 'peptide_count' (replace as needed)
df_columns = df.columns.tolist()
print('Available field @ids in this record set:', df_columns)

# Replace these field @ids with those found in your dataset
numeric_field_id = None
# Try a common ID or guess by field name
for c in df_columns:
    if 'count' in c.lower() or 'intens' in c.lower() or 'abundance' in c.lower():
        numeric_field_id = c
        break
if not numeric_field_id:
    numeric_field_id = df_columns[0]  # fallback
print(f"Selected numeric field: {numeric_field_id}\n")

# Pick a reasonable threshold for filtering, based on field
threshold = 1
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].nunique() > 5 else 1

# Filter
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Records with {numeric_field_id} > {threshold}: {len(filtered_df)}")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a likely categorical field
group_field_id = None
for c in df_columns:
    if 'type' in c.lower() or 'group' in c.lower() or 'category' in c.lower() or 'sample' in c.lower():
        group_field_id = c
        break

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped mean values by field {group_field_id}:")
    print(grouped_df.head())
else:
    print('No suitable group field found for groupby demonstration.')

## 5. Visualization
Plot the filtered and normalized numeric field.

You might try a histogram for the selected numeric field, or a boxplot/grouped bar chart if suitable groups were discovered.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Show histogram of normalized numeric field
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[f'{numeric_field_id}_normalized'].dropna(), bins=30, kde=True, color='skyblue')
plt.title(f'Distribution of normalized {numeric_field_id}')
plt.xlabel(f'{numeric_field_id} (normalized)')
plt.ylabel('Frequency')
plt.show()

# Grouped boxplot if possible
if group_field_id and group_field_id in filtered_df.columns:
    plt.figure(figsize=(12,4))
    sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, you:

- Loaded FAIR² dataset metadata and records using `mlcroissant`
- Explored available record sets and fields, always referencing by `@id`
- Extracted records into pandas DataFrames
- Applied basic filtering and normalization to numeric fields
- Visualized data distributions

Refer to the dataset documentation for specific field meaning and experiment notes. For further analysis, you might apply multivariate statistics to the normalized abundances, compare across sample groups, or prepare the data for machine learning or bioinformatics workflows.